In [3]:
import streamlit as st
import time
from app2.pipeline import SmartDocAssistant

# -----------------------------
# CONFIG
# -----------------------------
st.set_page_config(page_title="Smart Doc Assistant", layout="wide")

assistant = SmartDocAssistant()

# -----------------------------
# SESSION STATE
# -----------------------------
if "messages" not in st.session_state:
    st.session_state.messages = []

if "filename" not in st.session_state:
    st.session_state.filename = None

# -----------------------------
# TYPEWRITER FUNCTIONS
# -----------------------------
def typewriter_stream(container, stream, delay=0.01):
    """
    Character/stream-based typewriter effect
    """
    text = ""

    for chunk in stream:
        text += chunk
        container.markdown(text)
        time.sleep(delay)

    return text


def word_typewriter(container, stream, delay=0.03):
    """
    Word-based smoother typewriter effect
    """
    text = ""

    for chunk in stream:
        text += chunk

        words = text.split()
        display = ""

        for w in words:
            display += w + " "
            container.markdown(display)
            time.sleep(delay)

    return text

# -----------------------------
# SIDEBAR
# -----------------------------
st.sidebar.title("📄 Upload Document")

uploaded_file = st.sidebar.file_uploader("Upload PDF / DOCX / TXT")

if uploaded_file:

    if st.sidebar.button("⚙️ Process Document"):
        with st.spinner("Processing document..."):
            assistant.load_document_file(uploaded_file)
            assistant.process_document(uploaded_file.name)

            st.session_state.filename = uploaded_file.name

        st.sidebar.success("✅ Document processed!")

# -----------------------------
# MAIN UI
# -----------------------------
st.title("🧠 Smart Document Assistant")

if not st.session_state.filename:
    st.info("📌 Upload and process a document to start.")
    st.stop()

filename = st.session_state.filename

# -----------------------------
# QUICK ACTIONS
# -----------------------------
col1, col2, col3 = st.columns(3)

with col1:
    if st.button("📚 Full Summary"):
        with st.chat_message("assistant"):
            placeholder = st.empty()

            stream = assistant.summarize_document(filename)
            output = typewriter_stream(placeholder, stream)

            st.session_state.messages.append(
                {"role": "assistant", "content": output}
            )

with col2:
    chapter_input = st.text_input("📑 Chapter (e.g. chapter 2)")

with col3:
    if st.button("🔍 Chapter Summary"):
        if chapter_input:
            with st.chat_message("assistant"):
                placeholder = st.empty()

                stream = assistant.summarize_chapter(filename, chapter_input)
                output = typewriter_stream(placeholder, stream)

                st.session_state.messages.append(
                    {"role": "assistant", "content": output}
                )

# -----------------------------
# CHAT SECTION
# -----------------------------
st.divider()
st.subheader("💬 Chat with Document")

# show history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# input
user_query = st.chat_input("Ask anything about your document...")

if user_query:

    # save user message
    st.session_state.messages.append({"role": "user", "content": user_query})

    with st.chat_message("user"):
        st.write(user_query)

    with st.chat_message("assistant"):
        placeholder = st.empty()

        response_stream = assistant.ask(user_query, filename)

        output = typewriter_stream(placeholder, response_stream, delay=0.01)

    st.session_state.messages.append(
        {"role": "assistant", "content": output}
    )

/Users/akshayg/Documents/Projects/DOCSumerizer/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-14 20:20:48.441 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-14 20:20:48.489 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-14 20:20:48.489 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-04-14 20:20:48.489 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running 

✅ Assistant initialized
